name: 241211_data_analysis \
date: 02/04/2025 \
version: 2.2 \
github root: #google colab stream: version 2.0\
author: Justin Sankey, Johanna Ganglbauer \

**description**: Takes raw liquit chromatography mass spectroscopy (LCMS) data (exported table from SCIEX Analyst Software), computes recovery rates, method detection limits, and ratios of default channel to MS TOF channel. Generates plots and writes results to excel and creates long format table (.csv) for data publications.

**changes in comparison to previous version**:
*   'degenerate' in concentration declared as nan
*   removed 0.5 mL per sample from inputs
*   included final concentration table with numeric values only
*   sorted IPS in graph accordingly
*   included quantification from TOF channel as output (for now only concentrations, no recovery, mdl etc.)
*   replaced mass channel ratios with deviation of ion abundance ratios
*   within the MDL calculation: concentration < IDL are replaced with IDL - ensures MDL is never smaller than IDL, avoids high standard deviations for multiple concentrations below IDL
*   optional outlier detection for MDL calculation
*   concentration boxplots for MDL calculation
*   included retention time quality control in final table and long format table: calculated average shift between IS and compound in calibration and compared to shifts in samples for all the compounds
*   simplified final output and output tables with only four QAQC criteria: \
(i) retention time shift (RTS), (ii) below detection limit (BDL), (iii) poor recovery rate (Poor RR),
(iv) poor ion abundance ratio deviation (Poor IARD)

**what needs to be implemented**:
*   JB: read in different IDL files depending on sample matrix
*   JG: include EPA thresholds for recoveries
*   JG: check RTS not only for MSMS channel, but also for TOF MS channel
*   JG: further calculations with TOF MS channel (?)
*   SV: extract QAQC samples from researcher data and collect them somewhere (?)
*   TE: quality check concentrations as >10 % of initial blank values.
*   JG: Clean up 'old' recovery parts
*   JB: once enough data has been collected calculate 'default MDLs'
*   SV, JB: eventually include CCV in QAQC
*   **feel free to add your thoughts!**

**contact/help/complaints:** johanna.ganglbauer@uri.edu

# Specifying Inputs - Loading files and packages
The block below will load all python packages needed for the following analysis.

**Uncomment the first code block if you are using google colab and want to connect to the cloud**

In [ ]:
# connect google colab with your google drive
# from google.colab import drive
# drive.mount('/content/drive')

# import all needed packages
import os
import numpy as np
from openpyxl import load_workbook
from openpyxl.drawing.image import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pathlib as pali

# import pandas and suppress warnings
import pandas as pd
pd.options.mode.chained_assignment = None

**Enter in your input filepaths:**

(1) raw_data_filepaths is a list of files pointing to the exported table from the Sciex Analyst Software you want to analyze. Each file representing results from the core method must end with '_core', each file representing results from the extended method must end with '_extended'. \
(2) idl_filepath is a .csv file containing instrumentation detection limits (IDLs)

You can copy the filepath of each file on windows by right clicking on the file in your file explorer and choosing the option *Copy as Path*. \
The same works on Mac OS: Control-click or right-click on the file in Finder. Press the Option (Alt) key. Choose *Copy [filename] as Pathname*.

**Enter in your output filepaths:** \
When calling the script for the first time, two csv tables will be created which assist you in providing all the input parameters: \
(1) sample_input_parameter_filepath: contains a list of all samples in your input data and empty fields for you to provide the original sample volume or weight and allows you to decide if the sample should be considered for the evaluation of method detection limits. In the variable below you indicate where the sample_input_file should be saved to. Once the input file was created it will not be changed by the script. \
(2) threshold_input_parameter_filepath: contains a list of all injection standards and allows you to provide the valid range of recovery rates. In the variable below you indicate where the threshold_input_parameter should be saved to. Once the input file was created it will not be changed by the script.

Finally, the script will create a .csv file containing the most relevant data, as well as an excel file containing the most relevant results of the evaluation.\
(1) The variable processed_filepath indicates location and file prefix where your results should be saved to. \
(2) The variable plot_directory indictes location where your plots should be saved to.

**Make sure all your input and output filepaths are within apostrophes and start with an r. (r'...')** \
The appostrophes make python understand the variable is a string (text) and not a number.\
The r in front of the string (raw string) makes python interpret the backslashes in the file path correctly.

In [ ]:
### INPUT FILEPATHS
# raw data upload file path
raw_filepaths = [
    r'example_data_raw\taylor_sediment_1_core.txt',
    r'example_data_raw\taylor_sediment_2_core.txt',
]

# file paths for IDL and IQL data - not meant to be adopted
idl_filepath = r'example_data_raw/IDL_2024.csv'
threshold_input_parameter_filepath = r'example_data_raw/threshold_input_parameter.csv'

# input parameter filepath
sample_input_parameter_filepath = r'sample_input_parameter.csv'

### OUTPUT FILLEPATHS
# processed data output excel/csv file path - without .xlsx/.csv ending
processed_filepath =r'example_data_processed/test_data'

# directory to save plots to
plot_directory = r'example_figures'

**Enter in your parameters:**

(1) Indicate the maximal allowed ion abundance ratio deviation. (default from EPA 1633 is 50 %)\
(2) Indicate the maximal allowed difference of retention time between component and related IDA. (default from EPA 1633 is 0.1 min)\
(3) Provide a default value for the instrumentation detection limit (IDL) in case the component is not listed in the csv. \
(4) Choose if you want to consider detected outliers for MDL calculation (TRUE) or not (FALSE)
(5) Indicate which data you want to use to determine recovery rates: \
    (i)'core': only data from core method is used to determine recovery rates. This means peak integration on non-extracted standards (IPS) must be done for core method only. \
    (ii) 'extended': only data from from extended methode is used to determine recovery rates. This means peak integration of non-extracted standards (IPS) must be done for extended method only. \
    (iii) 'average': the average peak areas from core method and extended methods are used to determine recovery rates. This means peak integration of IPS peaks must be done for both core method and extended method.\
(6) Provide a concentration limit for the y-axis in plots. Default is 1.
(7) Provide a recovery rate limit for the y-axis in plots. Default is 1e3.

In [ ]:
# threshold for acceptance of percentage deviation from ion abundance ratio in sample compared to average ion abundance ratio in calibration midpoint.
allowed_ion_abundance_deviation = 30

# threshold for acceptance of deviation of retention time between PFAS component and related IDA
is_allowed_retention_time_deviation = 0.1

# idl default value
idl_unknown = 1e-3

# Set to True if outliers should not be considered for MDL calculation. Outliers are shown in the boxplot respectively.
exclude_outlier_for_mdl = False

# If a channel is available from both core and extended method, how should the component be chosen to determine Ion Abundance Ratio and Recovery Rates accordingly?
# available options: 'core', 'extended' and 'average'
channel_selection = 'core'

# sanity check - will produce an error if you indicated a variable recovery_data which is not allowed.
if channel_selection not in ['core', 'extended', 'average']:
    raise Exception("""
        Invalid input for variable recovery_data, use either 'core', 'extended', or 'average'.
                    """)

# will be used to identify standards by the column 'Component Name'
standard_identifiers = 'IDA|IPS|13C|d-|d3-|d5-|18O'

# will be used to identify calibration midpoint by the column 'Sample Name'
calibration_midpoint_identifier = 'CS5'

# limits for concentration values in plots
concentration_limit_plot = 2
# limits for recovery rate values in plots
recovery_limit_plot = 5e2

The following code block makes sure that the output folders exist and reads in data. Moreover, it sets the display options to show data tables in your console.

In [ ]:
# Ensure file path and folder path exist to write outputs to and create folders, if they do not exist
folder_path = os.path.dirname(processed_filepath)
if folder_path and not os.path.exists(folder_path):
    os.makedirs(folder_path)
processed_filepath_xlsx = processed_filepath + '.xlsx'
processed_filepath_csv = processed_filepath + '.csv'

if not os.path.exists(plot_directory):
    os.makedirs(plot_directory)

# Define columns of input which are needed for further processes:
columns_considered = [
    'Sample Name', 'Sample Index', 'Sample Comment', 'Sample Type',
    'Component Name',  'Component Group Name', 'Component Comment', 'IS Name',
    'Acquisition Date & Time', 'Used',
    'Calculated Concentration', 'Actual Concentration',
    'Reported Recovery', 'IDA Average Response Factor',
    'Area', 'Retention Time', 'IS Retention Time',
]

# Load input data files and put them all in one dataframe
data = pd.DataFrame()  # initialize empty data frames
sample_index = 0  # initialize Component Index
for file in raw_filepaths:
    # read in file
    if file[-4:] == '.csv':
        this_data = pd.read_csv(file, delimiter=',', encoding='utf-8', low_memory=False, header=0,)
    elif file[-4:] == '.txt':
        this_data = pd.read_csv(file, delimiter='\t', encoding='utf-8', low_memory=False, header=0,)
    else:
        raise ImportError('Raw input file paths must either be .csv or .txt files.')

    # increase component index to remain unique for multiple data frames
    this_data['Sample Index'] = this_data['Sample Index'] + sample_index
    sample_index += max(this_data['Sample Index'])
    check = this_data['Sample Index'].value_counts()

    # make sure each sample name ends with Ext for extended method and with Core for core method
    if file[-12:-4] == 'extended':
        mask_names = this_data['Sample Name'].str.endswith('Ext')
        this_data['Sample Name'][~mask_names] = [compound + ' Ext' for compound in this_data['Sample Name'][~mask_names].to_list()]

        # checks if extended data has 107 channels
        for index, elem in check[(check != 108)].items():
            print(f'The sample with Index {index} has {elem} channels.')

    elif file[-8:-4] == 'core':
        mask_names = this_data['Sample Name'].str.endswith('Core')
        this_data['Sample Name'][~mask_names] = [compound + ' Core' for compound in this_data['Sample Name'][~mask_names].to_list()]

        # checks if extended data has 107 channels
        for index, elem in check[(check != 142)].items():
            print(f'The sample with Index {index} has {elem} channels.')
    else:
        print('If you combine core method and extended method make sure your input file names end with _core and _extended respectively.')

    # append actual dataframe in list (this_data) to huge dataframe (data)
    if data.empty:
        data = this_data[columns_considered]  # initialize data in first step (when data is empty)
    else:
        data = pd.concat([data, this_data[columns_considered]], ignore_index=True)  # append to data

# display settings
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full width of columns

The following Code block cleans up data: \
 (i) replace strings: 'degenerate', 'no root' and 'NaN' with NaN
 (ii) replace strings: '<1 points' and '< 0' with 0
 (ii) make sure the component IPS-13C2_PFOA is not used as non extracted standard, replace it with IPS-13C4_PFOA, \
 (iii) make sure all _TOF MS channel names are labelled right.

In [ ]:
# Clean up 'Calculated Concentration' column
# set all strange strings to NaN
# set '<1 points' and '< 0' to 0
data['Calculated Concentration'] = data['Calculated Concentration'].replace(
    {'<1 points': 0, '< 0': 0, 'no root': np.nan, 'NaN': np.nan, 'degenerate': np.nan,}
    ).astype('float')

# Delete all '13C2_PFOA_TOF MS' channel rows if they are in the data set
data = data[~data['Component Name'].isin(['13C2_PFOA_TOF MS'])]

# Replace 'IPS-13C2_PFOA' values: optional, only when component name occurs
if any(data['Component Name'].isin(['IPS-13C4_PFOA'])):
    data['Component Group Name'] = data['Component Group Name'].replace('IPS-13C2_PFOA', 'IPS-13C4_PFOA')

    # Find rows where 'Component Group Name' is 'IPS-13C4_PFOA' (after replacement)
    mask = data['Component Group Name'] == 'IPS-13C4_PFOA'

    # Iterate through each of these rows and replace area in column
    for idx, row in data[mask].iterrows():
        sample_name = row['Sample Name']

        # Find the corresponding row with 'Component Name' == 'IPS-13C4_PFOA' and the same 'Sample Name'
        matching_row = data[(data['Component Name'] == 'IPS-13C4_PFOA') & (data['Sample Name'] == sample_name)]

        if not matching_row.empty:
            # Update the 'Area IPS' with the value from 'Area' in the matching row
            data.at[idx, 'Area IPS'] = matching_row['Area'].values[0]

# Correct channel names in original data (all of the TOF channels are labelled by _TOF MS, only 2 of them are labeled by only _TOF)
mask_names = data['Component Name'].str.endswith('_TOF')
data['Component Name'][mask_names] = [compound + ' MS' for compound in data['Component Name'][mask_names].to_list()]

# some have an underscore between TOF and MS, this is removed
mask_names = data['Component Name'].str.endswith('_TOF_MS')
data['Component Name'][mask_names] = [compound[:-3] + ' MS' for compound in data['Component Name'][mask_names].to_list()]

The following code block combines samples from the extended method and the core method and gives them the same sample index.

In [ ]:
# make sample index equal if one sample exist as both "Core Method" and "Extended Method"
sample_names = data['Sample Name'].unique()

# combine indices of Core Method and Extended Method if both are available
# initialize variables
detect = 0  # counts number of detected samples (core + extended is counted as one sample now)
index_mapper_sciex = {}  # dictionary for sample index - sample name linking
one_method_only_index = []  # initialize list of indices corresponding to only one method (either core or extended)
# loop over unique sample names
for sample_name in sample_names:

    # enter the following part if sample name is from extended method
    if sample_name[-3:] == 'Ext':  
        # check if corresponding core samples exist
        if sample_name[:-3] + 'Core' in sample_names:
            # get indices of all core samples the same name root
            sample_indices_core = data.loc[data['Sample Name'].isin([sample_name[:-3] + 'Core']), 'Sample Index'].value_counts().index
            # get indices of all extended samples with the same name
            sample_indices_extended = data.loc[data['Sample Name'].isin([sample_name]), 'Sample Index'].value_counts().index
            # loop over all tuples of core sample index and extended sample index
            for index_core, index_extended in zip(sample_indices_core, sample_indices_extended):
                # set sample index of core sample and extended sample to the same value (index of core sample)
                data.loc[data['Sample Index'].isin([index_core, index_extended]), 'Sample Index'] = index_core
                # save the link between new index and sample name to the dictionary index_mapper_sciex
                index_mapper_sciex[index_core] = sample_name[:-3]
                # count up the number of samples (extended + core)
                detect+=1

            # account for exceptions: more or less samples of core method than of extended one.
            # If there are less core samples than extended samples with the same name
            if len(sample_indices_core) < len(sample_indices_extended):
                print(f'At least one of the sample {sample_name} has no related Core Method. Find out if this is a problem.')
                # loop over extended indices without core counter part
                for index_extended in sample_indices_extended[len(sample_indices_core):]:
                    # save the link between index and sample name to the dictionary index_mapper_sciex
                    index_mapper_sciex[index_extended] = sample_name
                    # append index to samples with only one indices and count up
                    one_method_only_index.append(index_extended)
                    # count up the number of samples
                    detect+=1
            # If there are more core samples than extended samples with the same name
            elif len(sample_indices_core) > len(sample_indices_extended):
                print(f'At least one of the sample {sample_name} has no related extended Method. Find out if this is a problem.')
                # loop over core indices without extended counter part
                for index_core in sample_indices_core[len(sample_indices_extended):]:
                    # save the link between index and sample name to the dictionary index_mapper_sciex
                    index_mapper_sciex[index_core] = sample_name
                    # append index to samples with only one indices and count up
                    one_method_only_index.append(index_core)
                    # count up the number of samples
                    detect+=1
        # if no related core method is available for the extended method sample
        else:
            print(f'The sample {sample_name} has no related Core Method. Find out if this is a problem.')
            # get indices of all extended samples with the same name root
            sample_indices_extended = data.loc[data['Sample Name'].isin([sample_name]), 'Sample Index'].value_counts().index
            # loop over indices
            for index_extended in sample_indices_extended:
                # save the link between index and sample name to the dictionary index_mapper_sciex
                index_mapper_sciex[index_extended] = sample_name
                # append index to samples with only one indices and count up
                one_method_only_index.append(index_extended)
                detect+=1

    # enter the following part if sample name is from core method
    elif sample_name[-4:] == 'Core':
        # check if corresponding core samples is missing
        # if it is there it is already considered in the block above
        if not sample_name[:-4] + 'Ext' in sample_names:
            print(f'The sample {sample_name} has no related Extended Method. Find out if this is a problem.')
            # get indices of all core samples with the same name root
            sample_indices_core = data.loc[data['Sample Name'].isin([sample_name]), 'Sample Index'].value_counts().index
            # loop over indices
            for index_core in sample_indices_core:
                # save the link between index and sample name to the dictionary index_mapper_sciex
                index_mapper_sciex[index_core] = sample_name
                # append index to samples with only one indices and count up
                one_method_only_index.append(index_core)
                detect+=1
    else:
        print(f'Make sure your input data name ends either with _core or with _extended.')

In the following code block the order of PFAS compounds is conserved and the names are split to the (MS/MS) channel, screening fragmented masses (quantification), and the TOF channel, which screens for precursor masses (confirmation), which is used in rare cases (when matrix effects are expected in the fragmentation) for PFAS quantification.

If either channel is not available it is set to nan.

The following variables of relevance are created:
- compounds_msms: list of pfas compounds from the msms channel in the right order
- compounds_tof: list of pfas compounds from the tof channel in the right order
- idaips_msms: list of internal standards from the msms channel in the right order
- idaips_tof: list of internal standards from the tof channel in the right order

In [ ]:
# extract channel names from first sample and separate them into standards and non-standards
first_sample_id = data['Sample Index'].value_counts().index[0]  # index of first sample
compounds_filtered = data.loc[data['Sample Index'] == first_sample_id, 'Component Name']  # channel names of first sample
compounds_sorted = compounds_filtered[~compounds_filtered.str.contains(standard_identifiers)].to_list()  # channel names excluding IPS and IDA
ida_ips_sorted = compounds_filtered[compounds_filtered.str.contains(standard_identifiers)].to_list()  # channel names excluding IPS and IDA

# initialize lists
compounds_msms = []  # msms compounds
compounds_tof = []  # tof compounds
skip_compounds = []  # list of compounds already considered (can be skipped in followin iterations in loop)
# loop over all compounds from first sample
for component in compounds_sorted:
    if component in skip_compounds:  # skip iteration if compound was already considered in previous iterations
        continue
    if '_TOF MS' in component:  # in case compound is from TOF channel
        compounds_tof.append(component)  # add compound to TOF list
        skip_compounds.append(component)  # make sure the TOF compound is not considered more than once
        if component[:-7] in compounds_sorted:  # check if compound is available in corresponding MSMS channel
            compounds_msms.append(component[:-7])  # add msms compound to MSMS list
        else:
            compounds_msms.append(np.nan)  # add NaN to MSMS list if corresponding msms compound is not available
        skip_compounds.append(component[:-7])    # make sure the MSMS compound is not considered more than once
    else:  # in case compound is from MSMS channel
        compounds_msms.append(component)  # add compound to MSMS list
        skip_compounds.append(component)  # make sure the MSMS compound is not considered more than once
        if component + '_TOF MS' in compounds_sorted:  # check if compound is available in corresponding TOF channel
            compounds_tof.append(component + '_TOF MS')  # add tof compound to TOF list
        else:
            compounds_tof.append(np.nan)  # add NaN to TOF list if corresponding tof compound is not available
        skip_compounds.append(component + '_TOF MS')  # make sure the TOF compound is not considered more than once

# initialize and fill lists of sorted internal standards
ida_ips_msms = []  # msms internal standards
ida_ips_tof = []  # tof internal standards
skip_standards = []  # list of standards already considered (can be skipped in followin iterations in loop)
# loop over all standards from first sample
for standard in ida_ips_sorted:
    if standard in skip_standards: # skip iteration if standard was already considered in previous iterations
        continue
    if '_TOF MS' not in standard:  # in case standard is from MSMS channel
        ida_ips_msms.append(standard)  # add standard to MSMS list
        skip_standards.append(standard)  # make sure the MSMS standard is not considered more than once
        if standard[4:] + '_TOF MS' in ida_ips_sorted:  # check if standard is available in corresponding TOF channel
            ida_ips_tof.append(standard[4:] + '_TOF MS')  # add tof standard to TOF list
            skip_standards.append(standard[4:] + '_TOF MS')
        else:
            ida_ips_tof.append(np.nan)  # add NaN to TOF list if corresponding tof standard is not available
    else:  # in case standard is from TOF channel
        # in case standard is an IDA (differentiatiation only possible based on MSMS, as IDA and IPS label not available in TOF channel names)
        if 'IDA-' + standard[:-7] in ida_ips_sorted:
            ida_ips_tof.append(standard)  # add TOF standard to TOF list
            skip_standards.append(standard)  # make sure the TOF standard is not considered more than once
            if standard[4:] + '_TOF MS' in ida_ips_sorted:  # check if standard is available in corresponding TOF channel
                ida_ips_msms.append('IDA-' + standard[:-7])  # add MSMS standard to MSMS list
                skip_standards.append('IDA-' + standard[:-7])  # make sure the MSMS standard is not considered more than once
            else:
                ida_ips_msms.append(np.nan)  # add NaN to TOF list if corresponding tof standard is not available
        elif 'IPS-' + standard[:-7] in ida_ips_sorted:  # in case standard is an IPS
            ida_ips_tof.append(standard)  # add TOF standard to TOF list
            skip_standards.append(standard)  # make sure the TOF standard is not considered more than once
            if standard[4:] + '_TOF MS' in ida_ips_sorted:  # check if standard is available in corresponding TOF channel
                ida_ips_msms.append('IPS-' + standard[:-7])  # add MSMS standard to MSMS list
                skip_standards.append('IPS-' + standard[:-7])  # make sure the MSMS standard is not considered more than once
            else:
                ida_ips_msms.append(np.nan)  # add NaN to TOF list if corresponding tof standard is not available
        else:
            print(f'The standard: {standard} has no corresponding IDA or IPS in the default MS channel. It is ignored in the following calculations.')

ida_sorted = [ida_elem for ida_elem in ida_ips_msms if 'IDA' in ida_elem]  # extract IDAs from all msms standards
ips_sorted = [ips_elem for ips_elem in ida_ips_msms if 'IPS' in ips_elem]  # extract IPSs from all msms standards

The following code block separates data in calibration data, and quantification data.

In [ ]:
# Split data into quantification data, calibration data, and blanks for mdl calculation
calibration_only = data[(data['Sample Type'] == 'Standard')]
calibration_midpoint = calibration_only[(calibration_only['Sample Name'].str.contains(calibration_midpoint_identifier))]
quantification_blank = data[(data['Sample Type'] != 'Standard')]

In the following section a table with concentrations (masses) of PFAS for all samples and all analyzed PFAS components will be created and shown as well as a plot which shows the concentrations (masses).

Note: The LCMS quantifies ng/sample which is commonly referred to as concentration.

In [ ]:
# get rid of np.nan component names when used as index to create tables
pfas_components_msms = [component for component in compounds_msms if str(component) != 'nan']
pfas_components_tof = [component for component in compounds_tof if str(component) != 'nan']

pfas_mass = quantification_blank.pivot_table(
    index=('Sample Index'), columns='Component Name', values='Calculated Concentration', aggfunc='first', dropna=False,
)
pfas_mass.rename(index=index_mapper_sciex, inplace=True)
pfas_mass_msms = pfas_mass[pfas_components_msms]
pfas_mass_tof = pfas_mass[pfas_components_tof]

# write initial concentrations to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl') as writer:
    pfas_mass_msms.to_excel(writer, sheet_name='MSMS Mass (ng per Sample)')
    pfas_mass_tof.to_excel(writer, sheet_name='TOF Mass (ng per Sample)')

# previously done: plot PFAS concentrations
# from ipynb.fs.full.obsolete import plot_pfas_compounds
# plot_pfas_compounds(pfas_mass_data=pfas_mass_msms, pfas_compound_names=pfas_components_msms,
#        plot_directory=plot_directory, processed_filepath_xlsx=processed_filepath_xlsx)


In the following code block an empty csv table with a list of samples will be created for you to input \
(i) the sample name shown in the final table. If you are happy with the sample names currently in use you can make a copy.\
(ii) the sample volume/weight and unit. If you want to display your final results in ng/sample use 1 for sample quantity and 'sample' for sample unit. \
(iii) the samples considered for your mdl calculation set 'used for mdl calculation' to True or False accordingly.


In addition, an empty csv table with a list of extracted standards will be created for you to input the accepted range for the recovery rates. Per default the allowed recovery range from 50 % to 150 % is used - this will have to be finetuned accordingly.

**After the code was run for the first time, input all your parameters to the two files, save them and run the code again.**
Once the input files are created, they won't be changed by the script anymore.

In [ ]:
# checks if sample input parameter filepath already exists
if not pali.Path(sample_input_parameter_filepath).exists():
    # create a sample input parameter file if it did not exist before and stop the program if it did not.
    sample_input_data = pd.DataFrame(columns=[
        'sample name sciex data', 'sample name publication', 'volume/weight/number of samples', 'unit (e.g. g/mL/sample)', 'used for mdl calculation'
        ],
        index=quantification_blank['Sample Index'].unique()
        )
    sample_input_data['sample name sciex data'] = [index_mapper_sciex[sample_index] for sample_index in sample_input_data.index]
    sample_input_data['used for mdl calculation'] = False
    sample_input_data.to_csv(sample_input_parameter_filepath)
    del sample_input_data
    raise ImportError(
        f'The file {sample_input_parameter_filepath} was created for the first time.' + \
        'Adopt your sample quantities and units and select the samples you want to use for the evaluation of the method detection limit.'
    )

# checks if threshold input parameter filepath already exists
if not pali.Path(threshold_input_parameter_filepath).exists():
    # create a threshold input parameter file if it did not exist before.
    threshold_input_data = pd.DataFrame(columns=[
        'lower threshold for recoveries [%]', 'upper threshold for recoveries [%]'
        ],
        index=ida_sorted
        )
    threshold_input_data['lower threshold for recoveries [%]'] = 50
    threshold_input_data['upper threshold for recoveries [%]'] = 150
    threshold_input_data.to_csv(threshold_input_parameter_filepath)
    del threshold_input_data


The following code block loads your input parameters and input thresholds into data frames and dictionaries. \
**Make sure you input all the information in the provided input files.**

In [ ]:
sample_input_data = pd.read_csv(sample_input_parameter_filepath, index_col=[0])
index_mapper_publication = sample_input_data['sample name publication'].to_dict()
unit_mapper = sample_input_data['unit (e.g. g/mL/sample)'].to_dict()

recovery_thresholds = pd.read_csv(threshold_input_parameter_filepath, index_col=[0])
recovery_thresholds_dict = recovery_thresholds.to_dict()

# Visual Sanity Check 1:
The IPS Areas are compared in the calibration data and the samples themselves. The areas of the calibration data are internally multiplied by two in order two account for the differences in volume (calibration: 1 mL, quantification: 0.5 mL).

In most of the cases you add 4 ng internal standard to your 0.5 mL samples: the areas of the corresponding IPS should be in a comparable range for calibration data (left) and your samples (right) in the boxplots below.

**Doublecheck that the information you provided in the "Actual Concentration" column considering the amount of IPS and IDA you added to your samples is correct! This will affect your recovery rates. Usually you add 4 ng/(0.5 mL sample). Check the title of the right boxplot**.

In the following code there are no QAQC criteria, which flag samples with highly deviating IPS areas, it is your responsibility to check the ranges. It will affect the recovery rates, so it is a good idea to make sure that the areas are in comparable ranges, or you can explain why they are not.

In [ ]:
# Calculate IPS average area per compound in calibration data and plot it
calibration_only_ips = calibration_only[calibration_only['Component Name'].str.contains('IPS')]
calibration_only_ips['Area'] = calibration_only_ips['Area'] * 2 # normalize IPS area in calibration from 1 mL to 0.5 mL
quantification_blank_only_ips = quantification_blank[quantification_blank['Component Name'].str.contains('IPS')]

# evaluate mean per component
calibration_ips_area_averages = calibration_only_ips.groupby('Component Name')['Area'].mean()
quantification_blank_ips_area_averages = quantification_blank_only_ips.groupby('Component Name')['Area'].mean()

# evaluate IPS concentration in quantification
ips_concentration = quantification_blank_only_ips['Actual Concentration'].mean()

# create plot for comparison
image_path = os.path.join(plot_directory, 'ips_areas.png')
fig, ax = plt.subplots(ncols=2, figsize=(8, 8), sharey=True)
calibration_only_ips_grouped = calibration_only_ips.groupby('Component Name', sort=False)
calibration_only_ips_grouped.boxplot(column='Area', ax=ax[0], subplots=False)
ax[0].plot([np.nan] + calibration_ips_area_averages[ips_sorted].to_list(), color='red', linestyle='', marker="o", label='calibration average')
quantification_blank_only_ips_grouped = quantification_blank_only_ips.groupby('Component Name', sort=False)
quantification_blank_only_ips_grouped.boxplot(column='Area', ax=ax[1], subplots=False)
ax[1].plot([np.nan] + quantification_blank_ips_area_averages[ips_sorted].to_list(), color='red', linestyle='', marker="o", label='quantification average')
fig.suptitle('')
ax[0].set_title('Calibration: normalized to 4 ng/(0.5 mL)')
ax[1].set_title(f'Quantification: {round(ips_concentration, 0)} ng/(0.5 mL)')
ax[0].set_xticks(
    ticks=range(len(calibration_ips_area_averages) + 1),
    labels=[''] + ips_sorted, rotation=90
    )
ax[1].set_xticks(
    ticks=range(len(quantification_blank_ips_area_averages) + 1),
    labels=[''] + ips_sorted, rotation=90
    )
ax[0].set_ylabel('IPS area')
[this_ax.set_xlabel('') for this_ax in ax]
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# save IPS area comparison plot to excel file
workbook = load_workbook(processed_filepath_xlsx)
plot_sheet = workbook.create_sheet('MSMS IPS Area')

img = Image(os.path.join(plot_directory, 'ips_areas.png'))

cell_position = plot_sheet.cell(row=1, column=1).coordinate
plot_sheet.add_image(img, cell_position)

workbook.save(processed_filepath_xlsx)

In the following code block the data (calibration not included) is separated in the data junk used for mdl calculation,
and the remaining data (where the PFAS are quantified).

In [ ]:
indices_for_mdl_evaluation = sample_input_data[sample_input_data['used for mdl calculation']].index

if len(indices_for_mdl_evaluation) == 0:
    mdl_only = None
    quantification_only = quantification_blank
    print(f'Be careful, no samples have been collected for the MDL calculation.')
else:
    mdl_only = quantification_blank[quantification_blank['Sample Index'].isin(indices_for_mdl_evaluation)]
    quantification_only = quantification_blank[~quantification_blank['Sample Index'].isin(indices_for_mdl_evaluation)]

# QAQC Criterion 1: Retention Time Shift (RTS)
In each sample compare retention time of PFAS compound with retention time of corresponding IDA.
For Compounds with the molecular structure of the mass labeled internal standard being different than the molecular structure of the compound itself, the average time shift between the retention time is calculated from calibration midpoint sample and added to the tolerance criteria.

First Code Block below evaluates average retention time shift between analytical standard and corresponding internal standard from the calibration midpoint. Is set to zero if chemical structure of analytical standard and corresponding internal standard is the same. Shifts are saved to excel sheet 'RT shift'.

Second Code Block below provides functions to 'highlight' cells in excel red, based on boolean mask, as well as a function to rename samples (rows) in excel tables.

Third Code Block below calculates retention time shifts between analytical standard and corresponding internal standard and compares to limit, which is is_allowed_retention_time_deviation + calculated shift from first code block.

In [ ]:
# evaluate retention time of reference (=mid point calibration)
# extract relavant columns from calibration data
cal_mid_point = calibration_only[['Sample Name', 'Component Name', 'Retention Time', 'IS Name']]
# extract calibration midpoint from calibration data
cal_mid_point = cal_mid_point[cal_mid_point['Sample Name'].str.contains(calibration_midpoint_identifier)]
# calculate mean retention time from calibration data and save it to new data frame
cal_mid_point_rt = cal_mid_point.groupby('Component Name')['Retention Time'].mean()
# initial dataframe which saves acceptable retention time shifts from calibration midpoint 
rt_shift_df = pd.DataFrame({"RT":[], "IS RT":[], 'RT diff':[]})
# loops over all compounds of calibration midpoint
for index, retention_time in cal_mid_point_rt.items():
    # skip internal standards
    if any(substring in index for substring in ['IDA', 'IPS'] + standard_identifiers.split('|')):
        continue
    # get name of corresponding internal standard (IDA)
    is_name = cal_mid_point.loc[cal_mid_point['Component Name']==index, 'IS Name'].to_list()[0]
    # get retention time of corresponding inernal standard (IDA) in calibration mid point
    is_rt = cal_mid_point_rt[is_name]
    # if molecular structure (and naming) of compound and corresponding internal standard 
    # are the same, no additional shift in retention time is allowed
    if index in is_name:
        rt_shift_df.loc[index] = [retention_time, is_rt, 0]
    # if molecular structure (and naming) of compound and corresponding internal standard
    # are not the same, save addition shift in retention time from calibration midpoint
    else:
        rt_shift_df.loc[index] = [retention_time, is_rt, round(abs(retention_time - is_rt),2)]

# extract tof data and put TOF information from compound name to column header
rt_shift_tof = rt_shift_df.loc[pfas_components_tof]  # extract in right chemical order
rt_shift_tof.index = [item[0] for item in rt_shift_tof.index.str.split('_')]  # reindex with msms names
rt_shift_tof.columns = [elem + ' TOF' for elem in rt_shift_tof.columns]  # add 'TOF' to column names

# merge MSMS data with TOF data and save it to data frame rt_shift_excel
rt_shift_excel = pd.concat([rt_shift_df.loc[pfas_components_msms], rt_shift_tof], axis=1)

# write sample input data and retention time shifts to excel
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    sample_input_data.to_excel(writer, sheet_name='Sample Input Data')
    rt_shift_excel.to_excel(writer, sheet_name='RT shift')

In [ ]:
# function to color excel according to threshold values / mask
def highlight(data, flag):
    '''Sets all data elements to red background, when flag is True. '''
    if data.ndim == 1:  # Series from .apply(axis=0) or axis=1
        return ['background-color: red' if v else '' for v in flag]
    else:  # from .apply(axis=None)
        return pd.DataFrame(np.where(flag, 'background-color: red', ''),
                            index=data.index, columns=data.columns)

# function to replace sample names in excel sheet accordingly
def reindex_in_excel(filename: str, sheetname: str) -> None:
    """Function to rename index columns in excel."""
    # load excel file
    workbook = load_workbook(filename=filename)
    # open workbook
    sheet = workbook.active
    # get right sheet
    sheet = workbook[sheetname]
    # read in index column
    sample_index = [column.value for column in sheet['A']]
  
    # change index column values
    if sample_index[0] == 'Sample Index':
        for row_index in range(1, len(sample_index)):
            sheet.cell(row=row_index + 1, column=1).value = index_mapper_publication[sample_index[row_index]]
    
    #save the file
    workbook.save(filename=filename)
    workbook.close()

In [ ]:
# initialize new retention_time_data frame and select columns
selected_columns = [
    'Sample Name', 'Sample Index', 'Component Name', 'Retention Time', 'IS Retention Time'
]
retention_time_data = quantification_blank[selected_columns]

# select only channel names of precursor masses (confirmation) and fragmented masses (quantification)
retention_time_data = retention_time_data[retention_time_data['Component Name'].isin(compounds_msms + compounds_tof)]

# calculate deviation from retention time of PFAS component and related internal injection standard (IDA)
retention_time_data['Delta Retention Time IS'] = \
    round(retention_time_data['Retention Time'] - retention_time_data['IS Retention Time'], 2)

# initialize new column RTS indicating the limits
retention_time_data['RTS'] = False

# iterate over all samples and all compounds and check if retention time shifts are within limits
for (index, row) in retention_time_data.iterrows():
    # get additional time shift of internal standard from compound name and add to defined limit
    compound = row['Component Name']
    limit = is_allowed_retention_time_deviation + rt_shift_df.loc[compound, 'RT diff']
    # check if retention time shift is higher than limit
    if abs(row['Delta Retention Time IS']) > limit:
        retention_time_data.loc[index, 'RTS'] = True  # flag True if above limit

nan_flag = retention_time_data['Delta Retention Time IS'].isnull()  # save mask of NaN values
retention_time_data.loc[nan_flag, 'RTS'] = np.nan  # set mask values of nan retention time values to np.nan

for (compounds, sheet_name) in zip([pfas_components_msms, pfas_components_tof], ['MSMS RTS in min', 'TOF RTS in min']):
    # Put retention time data in pivot tables
    retention_time_is = retention_time_data.pivot_table(
        index=('Sample Index',), columns='Component Name', values='Delta Retention Time IS', aggfunc='first', dropna=False,
    )[compounds]
    retention_time_is_flag = retention_time_data.pivot_table(
        index=('Sample Index',), columns='Component Name', values='RTS', aggfunc='first', dropna=False,
    )[compounds]

    # color cells based on threshold values
    retention_time_is_styled = retention_time_is.style.apply(
        highlight, flag=retention_time_is_flag.fillna(value=False), axis=None
        )

    # Write to excel file
    with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
        retention_time_is_styled.to_excel(writer, sheet_name=sheet_name)
    # Use "publication" sample names in final table
    reindex_in_excel(filename=processed_filepath_xlsx, sheetname=sheet_name)

# Method Detection Limits
The following block computes method detection limits (MDL) based on average and standard deviation ($\sigma$) of PFAS concentrations in selected samples (process blanks, etc.). \
The code reads in instrument detection limits (IDL) for the PFAS compounds. For some compounds the IDL may not be included in the input files, in that case the default value is used. \
Before the MDL calculation, **all concentrations below the IDL are replaced with the IDL** in order to ensure that MDLs are never lower than IDLs, and that the standard deviation is low in case of multiple concentrations detected below IDL.\
In case no concentrations could be detected at all, the IDL is used as MDL. \

Within the Code the following definition of method detection limits is considered:\
$MDL_{PFAS} = average_{PFAS} + 3 \cdot \sigma_{PFAS}$

# Visual Sanity Check 2
The range of concentrations of selected blank samples for MDL calculations, IDL, and final MDL are depicted in the figure below. Outliers are shown and flagged. In case you do not include the outliers for the calculation of your MDLs, make sure that you report all blank samples with concentrations above the MDL. If the MDL for a certain PFAS compound is unreasonably high, you should not report/not trust the results for the corresponding compound and troubleshoot for contamination.

# Quality Control Criterion 2:
A new column 'Below Detection Limit (BDL)' is introduced, which indicates all 'Calculated Concentration' Values of PFAS quantification below the determined detection limits. These samples will be flagged as 'BDL' in the final concentration tables.

In [ ]:
### MDL calculation
# Initialize empty dataframe with IDL, relevant data for MDL calculation and MDL
mdl = pd.DataFrame(index=pfas_components_msms)
mdl['IDL'] = [np.nan] * len(mdl)
mdl['Mean Concentration'] = [np.nan] * len(mdl)
mdl['Std Concentration'] = [np.nan] * len(mdl)
mdl['MDL'] = [np.nan] * len(mdl)

# Load idl values from idl input file
idl = pd.read_csv(idl_filepath, index_col=0, low_memory=False, nrows=1)

# Write each idl value to idl column of MDL data frame and use default value idl_unknown if data is not available.
for row_index in mdl.index:
    if row_index in idl.columns:
        mdl.loc[row_index, 'IDL'] = idl[f'{row_index}'].to_list()[0]
    else:
        mdl.loc[row_index, 'IDL'] = idl_unknown
        print(f'No IDL available for {row_index}, default value of {idl_unknown} is used.')

# Calculate MDLs if samples for MDL calculations have been selected.
# Detects outliers and labels them
if not mdl_only is None:
    # select only MSMS compounds for mdl calculation
    blank_only_default = mdl_only[mdl_only['Component Name'].isin(pfas_components_msms)]
    # groups data selected for MDL calculations (blanks) by PFAS compound
    mdl_groups = blank_only_default.groupby('Component Name', sort=False)
    # initialize dataframe to collect outliers
    outlier_df = pd.DataFrame({'Compound Index': [], 'Concentration': [], 'Sample Name': []})
    label_value = 1  # initialize label value to assign labels of outiers in plot to right compound
    # loops over groups (PFAS compounds)
    for (title, group) in mdl_groups['Calculated Concentration']:
        relevant_idl = mdl.loc[title, 'IDL']
        # set all values below IDL to IDL
        group = group.apply(lambda x: x if x > relevant_idl else relevant_idl)
        # detect outliers in line with boxplot
        # outlier > 75 percentile + 1.5 (75 percentile - 25 percentile)
        outlier_threshold = group.quantile(0.75) + 1.5 * (
            group.quantile(0.75) - group.quantile(0.25)
            )
        # get raw data of outliers to create dataframe for labelling in plot
        potential_outliers = blank_only_default[blank_only_default['Component Name'].isin([title])]
        potential_outliers = potential_outliers[potential_outliers['Calculated Concentration'] > outlier_threshold]
        # if outliers are detected
        if not potential_outliers.empty:
            # save outliers to dataframe
            for _, row in potential_outliers.iterrows():
                outlier_df.loc[len(outlier_df)] = [label_value, row['Calculated Concentration'], row['Sample Name']]
            # if wished (exclude_outlier_for_mdl set to TRUE), replace concentrations above threshold with NaN
            if exclude_outlier_for_mdl:
                group = group.apply(lambda x: x if x <= outlier_threshold else np.nan)
        # calculate mean and standard deviation of concentrations in blank
        mdl.loc[title, 'Mean Concentration'] = group.mean()
        mdl.loc[title, 'Std Concentration'] = group.std()
        label_value += 1  # increase label value to label outliers in plot
    # calculate MDL
    mdl['MDL'] = round(mdl['Mean Concentration'] + 3 * mdl['Std Concentration'], 3)

# use MDL as detection threshold -> use IDL if MDL is not available (NaN)
mdl['Detection Limit'] = mdl['MDL']
mdl['Detection Limit'].fillna(mdl.IDL, inplace=True)

# write detection limits to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    mdl.to_excel(writer, sheet_name='Detection Limit')

In [ ]:
### MDL plots
# create plot to show details of MDL calculation
if not mdl_only is None:
    image_path = os.path.join(plot_directory, 'mdl.png')
    fig, ax = plt.subplots(figsize=(8, 8))

    # plot boxplot, mdl and idl
    mdl_groups.boxplot(column='Calculated Concentration', ax=ax, subplots=False)
    ax.plot([np.nan] + (mdl['IDL']).to_list(), color='blue', linestyle='', marker="o",)
    ax.plot([np.nan] + (mdl['MDL']).to_list(), color='red', linestyle='', marker="o",)
    for _, row in outlier_df.iterrows():
        if row['Concentration'] < 0.5:
            ax.text(row['Compound Index'], row['Concentration'], row['Sample Name'], rotation=90, va='bottom', ha='right')

    # axis, legend, descriptions, etc.
    ax.set_xticks(range(len(pfas_components_msms) + 1))
    ax.set_xticklabels([''] + pfas_components_msms, rotation=90)
    ax.set_ylim([0, concentration_limit_plot])
    fig.suptitle('')
    ax.set_title('')
    plt.ylabel('Concentration [ng/sample]')
    plt.xticks(rotation=90)
    box_patch = mpatches.Patch(color='blue', fill=False, label='concentrations')
    blue_patch = mpatches.Patch(color='blue', label='IDL')
    red_patch = mpatches.Patch(color='red', label='MDL')
    plt.legend(handles=[box_patch, red_patch, blue_patch])
    plt.savefig(image_path, bbox_inches='tight')
    plt.show()

    # save mdl box plot to excel file
    workbook = load_workbook(processed_filepath_xlsx)
    plot_sheet = workbook['Detection Limit']
    img = Image(os.path.join(plot_directory, 'mdl.png'))
    img.anchor = 'H1'
    plot_sheet.add_image(img)
    workbook.save(processed_filepath_xlsx)

In [ ]:
### Concentration values with MDL flags in excel
# initialize new concentration_data frame and select columns
selected_columns = [
    'Sample Name', 'Sample Index', 'Component Name', 'Calculated Concentration'
]

concentration_data = quantification_only[selected_columns]

# select only channel names of precursor masses (confirmation)
concentration_data = concentration_data[concentration_data['Component Name'].isin(compounds_msms)]

# initialize new column BDL indicating the limits
concentration_data['BDL'] = False

for (index, row) in concentration_data.iterrows():
    # get detection limit from compound name
    compound_name = row['Component Name']
    detection_limit = mdl.loc[compound_name, 'Detection Limit']
    # check if concentration is below detection limit
    if abs(row['Calculated Concentration']) < detection_limit:
        concentration_data.loc[index, 'BDL'] = True  # flag True if above limit

concentration_data['Calculated Concentration'] = round(concentration_data['Calculated Concentration'], 4)
nan_flag = concentration_data['Calculated Concentration'].isnull()  # save mask of NaN concentration values
concentration_data.loc[nan_flag, 'BDL'] = np.nan  # set mask values of nan concentrtion values to np.nan

# Put concentration data in pivot tables
concentration_data_pivot = concentration_data.pivot_table(
    index=('Sample Index',), columns='Component Name', values='Calculated Concentration', aggfunc='first', dropna=False,
)[pfas_components_msms]
concentration_data_pivot_flag = concentration_data.pivot_table(
    index=('Sample Index',), columns='Component Name', values='BDL', aggfunc='first', dropna=False,
)[pfas_components_msms]

# color cells based on threshold values
concentration_data_pivot_styled = concentration_data_pivot.style.apply(
    highlight, flag=concentration_data_pivot_flag.fillna(value=False), axis=None
    )

# Write to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    concentration_data_pivot_styled.to_excel(writer, sheet_name="MSMS Concentrations in ng MDL")
# Use "publication" sample names in final table
reindex_in_excel(filename=processed_filepath_xlsx, sheetname="MSMS Concentrations in ng MDL")

# Quality Control Criterion 3: Recovery Rates
To avoid misunderstandings, in the following two abbreviations are extensively used:
- IDA: isotope dilution analysis, also known as SS=surrogate standard or EIS=extracted internal standard
- IPS: isotope performance standard, also known as IS=injection standard or NIS=non-extracted internal standard

**The following two blocks calculate response factors from calibration data:**\
ratio of (i) calculated area of IDA * **actual concentration of IPS** and (ii) calculated area of IPS * actual concentration of IDA. \
The data is saved to an excel file and a boxplot of response factors is created.

Note: The response factor calculation within sciex uses the ratio of (i) calculated area of IDA and (ii) calculated area of IPS * actual concentration of IDA. \
As the concentration of IPS is missing in the calculation, the response factors deviate by a factor of 4, which is the actual concentration of IPS in the calibration data. \
Note: The column 'Actual Concentration' provides information on ng of internal standard added per sample.

**The third and forth code block calculate recovery rates for each IDA compound in each sample:**
$$
recovery~rate = \frac{\frac{area_{IDA~sample}~\cdot~concentration_{IPS~sample}}{area_{IPS~sample}~\cdot~concentration_{IDA~sample}}}{average(\frac{area_{IDA~calibration}~\cdot~concentration_{IPS~calibration}}{area_{IPS~calibration}~\cdot~concentration_{IDA~calibration}})} = \frac{IDA-IPS~ratio}{response~factor}
$$

In [ ]:
# define funtion which calculates the IDA IPS ratio.
# challenge - search right IPS row indicated in the Component Group Name of IDA.
def calculate_ida_ips_ratio(data: pd.DataFrame, column_name:str, ) -> pd.DataFrame:
    """Calculates IDA area times IPS concentration divided by IPS area times IDA concentration and save the results in the indicated column.

    :param data: Entire data junk (including all rows and the following columns:
    Component Name, Sample Index, Component Group Name, Actual Concentration, Area, and IDA Average Response Factor
    :type data: pd.DataFrame
    :param column_name: name of column, the calculated ratio should be saved to
    :type column_name: str
    :return: Data junk only containing IDA rows with the corresponding ratio saved to new column
    :rtype: pd.DataFrame
    """
    # select only ida rows from input data
    data_only_ida = data[data['Component Name'].str.contains('IDA')]
    # initialize new column names
    data_only_ida[[f'{column_name}', 'IPS Area', 'IPS Concentration']] = np.nan

    # calculate recovery rate for every component, end every sample
    for row_index in data_only_ida.index:
        sample_index = data_only_ida.loc[row_index, 'Sample Index']
        ips_channel_name = data_only_ida.loc[row_index, 'Component Group Name']

        sample_name = data_only_ida.loc[row_index, 'Sample Name']
        corresponding_ips_area_row = data[(
            (data['Sample Index'] == sample_index) &
            (data['Component Name'] == ips_channel_name) &
            (data['Sample Name'] == sample_name)
            )]
        data_only_ida.loc[row_index, 'IPS Area'] = corresponding_ips_area_row['Area'].iloc[0]
        data_only_ida.loc[row_index, 'IPS Concentration'] = corresponding_ips_area_row['Actual Concentration'].iloc[0]
        data_only_ida.loc[row_index, f'{column_name}'] = \
            (data_only_ida.loc[row_index, 'Area'] * corresponding_ips_area_row['Actual Concentration'].iloc[0]) \
            / (corresponding_ips_area_row['Area'].iloc[0] * data_only_ida.loc[row_index, 'Actual Concentration'])
    return data_only_ida

In [ ]:
# Extract values of IDAs and IPS
# Save basic sample information as well as areas of intensity peak, actual concentration and Component Group Name.
# The 'Component Group Name' is useful to assoiciate the right IPS to each IDA.

# calucluate IDA IPS ratio to compute response factors with calibration data
calibration_only_ida = calculate_ida_ips_ratio(
    data=calibration_only, column_name='Response Factor Mean',
    )

# create data frame with this response factor calculation (from scratch), the standard deviation and the original values evaluated by Sciex,
response_factor = calibration_only_ida.groupby('Component Name', as_index=False)['Response Factor Mean'].mean()
response_factor['Response Factor Std'] = calibration_only_ida.groupby('Component Name')['Response Factor Mean'].std().to_list()
response_factor['Response Factor Sciex'] = calibration_only_ida.groupby('Component Name')['IDA Average Response Factor'].mean().to_list()
response_factor.index = response_factor['Component Name']
response_factor = response_factor.drop(columns=['Component Name']).reindex(ida_sorted)

# write response factor to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    response_factor.to_excel(writer, sheet_name='Response Factor')

# create and save response factor box plots
image_path = os.path.join(plot_directory, 'response_factors.png')
fig, ax = plt.subplots(figsize=(8, 8))

response_factor_grouped = calibration_only_ida.groupby('Component Name', sort=False)
response_factor_grouped.boxplot(column='Response Factor Mean', ax=ax, subplots=False)

# calibration_only_ida.boxplot(column='Response Factor Mean', by='Component Name', ax=ax,)
ax.plot([np.nan] + (response_factor['Response Factor Sciex'] * 4).to_list(), color='red', linestyle='', marker="o",)
ax.plot([np.nan] + (response_factor['Response Factor Sciex']).to_list(), color='blue', linestyle='', marker="o",)
ax.set_xticks(range(len(response_factor_grouped) + 1))
ax.set_xticklabels([''] + ida_sorted, rotation=90)
fig.suptitle('')
ax.set_title('')
plt.ylabel('Response Factor (IDA area/IPS area)')
plt.xticks(rotation=90)
box_patch = mpatches.Patch(color='blue', fill=False, label='data')
blue_patch = mpatches.Patch(color='blue', label='RF Sciex')
red_patch = mpatches.Patch(color='red', label='4 * RF Sciex')
plt.legend(handles=[box_patch, red_patch, blue_patch])
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# save response factor box plot to excel file
workbook = load_workbook(processed_filepath_xlsx)
plot_sheet = workbook['Response Factor']
img = Image(os.path.join(plot_directory, 'response_factors.png'))
img.anchor = 'F1'
plot_sheet.column_dimensions['F'].width = img.width / 6
plot_sheet.add_image(img)
workbook.save(processed_filepath_xlsx)

Recovery rate computed within the SCIEX software is not normalized by concentration. So the recovery rate calculated by SCIEX reads:
$$
recovery~rate = \frac{\frac{area_{IDA~sample}}{area_{IPS~sample}}}{average(\frac{area_{IDA~calibration}}{area_{IPS~calibration}})}
$$

The uncertainty of the recovery rate is indicated by a maximum error method - for now, you can just ignore this part...
$$
\Delta recovery~rate = \Delta response~factor \cdot \frac{ratio}{response~factor^{2}} + \Delta ratio \cdot \frac{1}{response~factor}
$$

In [ ]:
# function to determine if recovery rates are within given limit
def flag_poor_recovery(data: pd.DataFrame, checked_column_name: str, new_column_name: str, columns_sorted: list[str]) -> tuple[pd.DataFrame, None]:
    """Checks if recovery rates are in defined threshold limits and appends check results as boolean column to data.
    Additionally formats table for recovery rate.

    :param data: Entire data junk including all rows and the following columns:
    'Component Name', 'Sample Index', f'{checked_column_name}'
    :type data: pd.DataFrame
    :param checked_column_name: name of column which is used to check if values are in the given limits.
    :type checked_column_name: str
    :param new_column_name: name of column, the check results should be saved to.
    :type new_column_name: str
    :param columns_sorted: List indicating the order of columns of pivot tables
    :type columns_sorted: list[str]
    :return: Original data junk with the new column (ckeck results) appended + formated pivot table containing recovery rates.
    :rtype: tuple[pd.DataFrame, pd.DataFrame.style.Styler]
    """
    data[f'{new_column_name}'] = np.nan
    for component_ida in data['Component Name'].unique():
        lower_threshold_recovery = recovery_thresholds_dict['lower threshold for recoveries [%]'][component_ida]
        upper_threshold_recovery = recovery_thresholds_dict['upper threshold for recoveries [%]'][component_ida]
        data.loc[
            (data['Component Name'] == component_ida) & 
            (~data[f'{checked_column_name}'].isnull())
        , f'{new_column_name}'] = True
        data.loc[
            (data['Component Name'] == component_ida) &
            (data[f'{checked_column_name}'] > lower_threshold_recovery) &
            (data[f'{checked_column_name}'] < upper_threshold_recovery)
        , f'{new_column_name}'] = False

    recovery_pivot = data.pivot_table(
        index=('Sample Index',), columns='Component Name', values=f'{checked_column_name}', aggfunc='mean', dropna=False,
    )[columns_sorted]

    poor_recovery_pivot = data.pivot_table(
        index=('Sample Index',), columns='Component Name', values=f'{new_column_name}', aggfunc='first', dropna=False,
        )[columns_sorted]

    # color cells based on recovery values
    recovery_styled = recovery_pivot.style.apply(highlight, flag=poor_recovery_pivot.fillna(value=True), axis=None)

    return data, recovery_styled


In [ ]:
# Select ida rows from quantification data and calculate ida ips ratio
# function is defined in previous block
quantification_ida = calculate_ida_ips_ratio(
    data=quantification_blank, column_name="IDA-IPS Ratio",
    )

recovery_table = pd.DataFrame(columns=[
    'Sample Index', 'Component Name', 'IDA-IPS Ratio',
    'Response Factor Mean', 'Recovery Rate',
    ])

# delete IDA and IPS values from one method if 'core' or 'extended' is selected,
# and use the average of both methods in the alternative case.
index = 0

for sample_index in quantification_ida['Sample Index'].unique():
    for component_ida in quantification_ida['Component Name'].unique():
        selected_sample_ida = quantification_ida.loc[(
            (quantification_ida['Sample Index'] == sample_index) & 
            (quantification_ida['Component Name'] == component_ida)
        ), :]
        response_factor_mean = response_factor.loc[component_ida, 'Response Factor Mean']
        if len(selected_sample_ida.index) == 0:
            print(f'Component {component_ida} is not available for sample {sample_index}')
            ida_ips_ratio_mean = np.nan
        elif len(selected_sample_ida.index) == 1:
            ida_ips_ratio_mean = selected_sample_ida['IDA-IPS Ratio'].values[0]
        else:
            if channel_selection == 'average':
                ida_ips_ratio_mean = selected_sample_ida['IDA-IPS Ratio'].mean()
            elif channel_selection == 'core':
                ida_ips_ratio_mean = selected_sample_ida.loc[
                    selected_sample_ida['Sample Name'].str.contains('Core'), 'IDA-IPS Ratio'
                    ].values[0]
            elif channel_selection == 'extended':
                ida_ips_ratio_mean = selected_sample_ida.loc[
                    selected_sample_ida['Sample Name'].str.contains('Ext'), 'IDA-IPS Ratio'
                    ].values[0]
            else:
                raise Exception("""
                Invalid input for variable recovery_data, use either 'core', 'extended', or 'average'.
                            """)

        recovery_rate = 100 * ida_ips_ratio_mean / response_factor_mean
        recovery_table.loc[index] = [
            sample_index, component_ida, ida_ips_ratio_mean,
            response_factor_mean, recovery_rate,
            ]
        index +=1

# check if recovery rate is within indicated limit for each ida and save check results to 'Poor Recovery column
recovery_table, recovery_styled = flag_poor_recovery(data=recovery_table, checked_column_name='Recovery Rate', new_column_name='Poor Recovery', columns_sorted=ida_sorted)

# Write recovery rate to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    recovery_thresholds.to_excel(writer, sheet_name='Recovery Thresholds')
    recovery_styled.to_excel(writer, sheet_name='MSMS Recovery Rate')
    
reindex_in_excel(filename=processed_filepath_xlsx, sheetname='MSMS Recovery Rate')

The following code cells generate plots of recovery rates over all samples.

In [ ]:
# Box plot for recovery rates
image_path = os.path.join(plot_directory, 'recovery_rates_box.png')
fig, ax = plt.subplots(figsize=(8, 8))
recovery_grouped = recovery_table.groupby('Component Name', sort=False)
recovery_grouped.boxplot(column='Recovery Rate', ax=ax, subplots=False)
ax.set_ylim([0, recovery_limit_plot])
ax.set_xticks(range(len(recovery_grouped) + 1))
ax.set_xticklabels([''] + ida_sorted, rotation=90)
fig.suptitle('')
ax.set_title('')
plt.ylabel('Recovery Rate')
plt.xticks(rotation=90)
plt.legend()
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# plot data as points for recovery rates:
plot_data = recovery_table.groupby('Sample Index')  # group data for plotting
cmap = plt.cm.get_cmap('tab20', len(plot_data)) # initialize colours
image_path = os.path.join(plot_directory, 'recovery_rates.png')  # set path for figure

fig, ax = plt.subplots(figsize=(8, 8))
for index, (title, group) in enumerate(plot_data):
    group.set_index(group['Component Name'], inplace=True)
    group.drop_duplicates(keep='first', inplace=True)
    group.plot(
        y='Recovery Rate', ax=ax, marker='.', linestyle='None', label=index_mapper_publication[title],
        grid=True, color = cmap(index),
    )
ax.set_ylim([0, recovery_limit_plot])
fig.suptitle('')
ax.set_xticks(range(len(group)))
ax.set_xticklabels(group['Component Name'], rotation=90)
ax.set_title('')
plt.xticks(rotation=90)
plt.ylabel('Recovery Rate')
plt.legend(loc='center right', bbox_to_anchor=(1.4, 0.5))
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# save recovery rate plot to excel file
workbook = load_workbook(processed_filepath_xlsx)
plot_sheet = workbook.create_sheet('MSMS Recovery Rates Plots')

img1 = Image(os.path.join(plot_directory, 'recovery_rates_box.png'))
img1.anchor = 'A1'
plot_sheet.column_dimensions['A'].width = img1.width / 6
plot_sheet.row_dimensions[1].height = img.height
plot_sheet.add_image(img1)

img2 = Image(os.path.join(plot_directory, 'recovery_rates.png'))
img2.anchor = 'B1'
plot_sheet.column_dimensions['B'].width = img2.width / 6
plot_sheet.add_image(img2)

workbook.save(processed_filepath_xlsx)

# Quality Criterion 4: Ion abundance ratios
The following block evaluates ion abundance ratios from midpoint calibration data, computes the average and plots the results. \
The ion abundance ratio (IAR) is caluclated for each PFAS compound and each standard as follows:

$IAR_{PFAS} = \frac{Area-TOF_{PFAS}}{Area-MSMS_{PFAS}}$

In [ ]:
# Assign TOF channel to each MS MS channel and caluclate ion abundance ratio. 
# define funtion which calculates the ion abundance ratio
# challenge - search right TOF corresponding to MS MS.
def calculate_ion_abundance_ratio(pfas_components_tof: list[str], pfas_components_msms: list[str], data:pd.DataFrame) -> pd.DataFrame:
    """Calculate ion abundance ratio of a given data set.

    :param pfas_components_tof: List of all PFAS compounds having a TOF_MS channel in the right order.
    :type pfas_components_tof: list[str]
    :param pfas_components_msms: List of all PFAS compounds having an MSMS channel in the right order.
    :type pfas_components_msms: list[str]
    :param data:  Entire data junk (including all rows and the following columns:
    Component Name, Sample Index, Sample Name, Area
    :type data: pd.DataFrame
    :return:  Data junk containing area of tof component and ion abundance ratio for msms channels as new column.
    :rtype: pd.DataFrame
    """
    # Initialize new columns and set them to NaN per default
    data['Ion Abundance Ratio'] = np.nan
    data['TOF Area'] = np.nan

    # get index of msms component of exact same sample and tof component of exact same sample
    for sample_index in data['Sample Index'].unique():
        for (msms_component, tof_component) in zip(pfas_components_msms, pfas_components_tof):
            msms_row_index = data[(
                (data["Sample Index"] == sample_index) &
                (data['Component Name'] == msms_component))].index.to_list()
            tof_row_index = data[(
                (data["Sample Index"] == sample_index) &
                (data['Component Name'] == tof_component))].index.to_list()

            # find right msms channel row in case two components are availeble (core vs. extended method)
            if len(msms_row_index) == 0:
                print(f'Component {msms_component} is not available for sample {index_mapper_sciex[sample_index]}.')
            elif len(msms_row_index) == 1:
                msms_area = data.loc[msms_row_index[0], 'Area']
            elif len(msms_row_index) == 2:
                relevant_rows = data.loc[msms_row_index]
                if channel_selection == 'average':
                    msms_area = relevant_rows['Area'].mean()
                    print(relevant_rows, msms_area)
                elif channel_selection == 'core':
                    msms_area = relevant_rows.loc[
                        relevant_rows['Sample Name'].str.contains('Core'), 'Area'
                        ].values[0]
                elif channel_selection == 'extended':
                    msms_area = relevant_rows.loc[
                        relevant_rows['Sample Name'].str.contains('Ext'), 'Area'
                        ].values[0]
            else:
                print('More than two exact same channels found in the same sample, that is very suspicious')

            # find right tof channel row in case two components are availeble (core vs. extended method)
            if len(tof_row_index) == 0:
                print(f'Component {tof_component} is not available for sample {index_mapper_sciex[sample_index]}.')
            elif len(tof_row_index) == 1:
                tof_area = data.loc[tof_row_index[0], 'Area']
            elif len(tof_row_index) == 2:
                relevant_rows = data.loc[tof_row_index]
                if channel_selection == 'average':
                    tof_area = relevant_rows['Area'].mean()
                    print(relevant_rows, tof_area)
                elif channel_selection == 'core':
                    tof_area = relevant_rows.loc[
                        relevant_rows['Sample Name'].str.contains('Core'), 'Area'
                        ].values[0]
                elif channel_selection == 'extended':
                    tof_area = relevant_rows.loc[
                        relevant_rows['Sample Name'].str.contains('Ext'), 'Area'
                        ].values[0]
            else:
                print('More than two exact same channels found in the same sample, that is very suspicious')
    
            # calculate IAR if both msms channel and tof ms channel are available
            if not (np.isnan(msms_area) or np.isnan(tof_area)):
                data.loc[msms_row_index, 'TOF Area'] = tof_area
                data.loc[msms_row_index, 'Ion Abundance Ratio'] = tof_area / msms_area
    return data

In [ ]:
# select relavant columns of calibration midpoint data
selected_columns = [
    'Sample Index', 'Sample Name', 'Component Name', 'Area',
    ]
mass_channel_area_calibration_midpoint = calibration_midpoint[selected_columns]

# calculate ion abundance ratios and drop tof channel rows afterwards
ion_abundance_ratio_calibration_midpoint = calculate_ion_abundance_ratio(
    pfas_components_msms=pfas_components_msms + ida_ips_msms, pfas_components_tof=pfas_components_tof + ida_ips_tof,
    data=mass_channel_area_calibration_midpoint
)
ion_abundance_ratio_calibration_midpoint = ion_abundance_ratio_calibration_midpoint[
    ~ion_abundance_ratio_calibration_midpoint['Component Name'].str.contains('_TOF MS')
]

# group ion abundance ratios in calibration midpoint by component
ion_abundance_ratio_calibration_midpoint_grouped = ion_abundance_ratio_calibration_midpoint.groupby(
    'Component Name', as_index=True, sort=False)

# evaluate mean of groups - ion abundance ratio per compound in calibration midpoint
ion_abundance_ratio_calibration_midpoint_mean = ion_abundance_ratio_calibration_midpoint_grouped['Ion Abundance Ratio'].mean()

# write ion abundance ratio in calibration midpoint to excel
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    ion_abundance_ratio_calibration_midpoint_mean.to_excel(
        writer, sheet_name=f'{calibration_midpoint_identifier} IAR'
        )

# create and save response factor box plots
image_path = os.path.join(plot_directory, 'midpoint_ion_abundance_ratio.png')
fig, ax = plt.subplots(figsize=(16, 8))
ion_abundance_ratio_calibration_midpoint_grouped.boxplot(column='Ion Abundance Ratio', ax=ax, subplots=False)

# calibration_only_ida.boxplot(column='Response Factor Mean', by='Component Name', ax=ax,)
ax.set_xticks(range(len(ion_abundance_ratio_calibration_midpoint_grouped ) + 1))
ax.set_xticklabels([''] + pfas_components_msms + ida_ips_msms, rotation=90)
fig.suptitle('')
ax.set_title('')
plt.ylabel('Ion Abundance Ratio(TOF channel area / MS-MS channel area)')
plt.xticks(rotation=90)
plt.savefig(image_path, bbox_inches='tight')
plt.show()

# save image to excel
workbook = load_workbook(processed_filepath_xlsx)
plot_sheet = workbook[f'{calibration_midpoint_identifier} IAR']
img = Image(image_path)
img.anchor = 'D1'
plot_sheet.column_dimensions['D'].width = img.width / 6
plot_sheet.add_image(img)
workbook.save(processed_filepath_xlsx)

The following code block evaluates ion abundance ratios in the quantification data and compares them to the ion abundance ratios found in the calibration midpoint (CSM) to evaluate the percentage deviation:

$IAR~deviation_{sample, PFAS} = 100 \cdot (\frac{IAR_{sample, PFAS}}{mean(IAR_{CSM,PFAS})} - 1)$

In [ ]:
# initialize new ion abundance ratio data frame and select columns
ion_abundance_ratio = quantification_blank[selected_columns]

# calculate ion abundance ratio for all compounds and standards and delete rows from TOF channel afterwards
ion_abundance_ratio = calculate_ion_abundance_ratio(
    pfas_components_msms=pfas_components_msms + ida_ips_msms, pfas_components_tof=pfas_components_tof + ida_ips_tof,
    data=ion_abundance_ratio
)
ion_abundance_ratio = ion_abundance_ratio[
    ~ion_abundance_ratio['Component Name'].str.contains('_TOF MS')
]

# Initialize new Ion Abundance Ratio Deviation column and set to NaN per default.
ion_abundance_ratio['Ion Abundance Ratio Deviation [%]'] = np.nan

for sample_index in ion_abundance_ratio['Sample Index'].unique():
    for compound in ion_abundance_ratio['Component Name'].unique():
        selected_index = ion_abundance_ratio[(
            (ion_abundance_ratio['Sample Index'] == sample_index) & 
            (ion_abundance_ratio['Component Name'] == compound)
        )].index.to_list()
        if len(selected_index) == 1:
            ion_abundance_ratio_calibration_midpoint = ion_abundance_ratio_calibration_midpoint_mean.loc[compound]
            ion_abundance_ratio.loc[selected_index[0], 'Ion Abundance Ratio Deviation [%]'] = \
                100 * (ion_abundance_ratio.loc[selected_index[0], 'Ion Abundance Ratio'] / ion_abundance_ratio_calibration_midpoint - 1)
        else:
            print(sample_index, index_mapper_sciex[sample_index])
# Introduce new column where everything above or below 50 % deviation is flagged
nan_flag = ion_abundance_ratio['Ion Abundance Ratio Deviation [%]'].isnull()
ion_abundance_ratio[f'Ion Abundance Ratio Deviation > {allowed_ion_abundance_deviation}'] = abs(ion_abundance_ratio['Ion Abundance Ratio Deviation [%]']) > allowed_ion_abundance_deviation
ion_abundance_ratio.loc[nan_flag, f'Ion Abundance Ratio Deviation > {allowed_ion_abundance_deviation}'] = np.nan

# Put channel ratio in pivot table
ion_abundance_ratio_table = ion_abundance_ratio.pivot_table(
    index=('Sample Index',), columns='Component Name', values='Ion Abundance Ratio Deviation [%]', aggfunc='mean', dropna=False,
)[pfas_components_msms + ida_ips_msms]
ion_abundance_ratio_flag = ion_abundance_ratio.pivot_table(
    index=('Sample Index',), columns='Component Name', values=f'Ion Abundance Ratio Deviation > {allowed_ion_abundance_deviation}', aggfunc='first', dropna=False,
)[pfas_components_msms + ida_ips_msms]

# color cells based on threshold values
ion_abundance_ratio_styled = ion_abundance_ratio_table.style.apply(
    highlight, flag=ion_abundance_ratio_flag.fillna(value=False), axis=None
    )

# Write to excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    ion_abundance_ratio_styled.to_excel(writer, sheet_name='IAR Deviation in %')
reindex_in_excel(filename=processed_filepath_xlsx, sheetname='IAR Deviation in %')

# Ouputs and data merging

In the following code block all relevant data and parameters are combined in one common dataframe quantification_pfas_default, which has one row for each (MSMS) PFAS compound in each sample: \
(1) The recovery rate is assigned to each (MSMS) PFAS compound. Each PFAS compound has a corresponding IDA - the recovery rate is deduced from the recovery rate of the corresponding IDA. The assignment works for each sample by using the 'IS Name' column which provides information on which IDA standard is associated to which PFAS. In adddition, a boolean column is appended, indicating if the recovery rates are in the allowed limits.\
(2) detection limits are appended to the table as well as a boolean column indicating if the detected values are below the threshold. \
(3) basic information about the sample are included in the final data frame. \
(4) ion abundance ratios deviations are included, as well as a boolean column inidcating if the QAQC criteria are met.
(5) retention time differences are included in the final table with respect to internal standards and the same compounds in the calibration midpoint. For both options a boolean column inidcating if the QAQC criteria are met is appended.

In [ ]:
# Initialize data frame for following assignments, select only needed columns and use reasonable naming.
selected_columns = [
    'Sample Name', 'Sample Index', 'Acquisition Date & Time', 'Component Name', 'Calculated Concentration',
    'Area', 'IS Name',
    ]
quantification_pfas_default = quantification_only[selected_columns]
quantification_pfas_default.rename(columns={'IS Name': 'IDA Name'}, inplace=True)

# Get only PFAS default channels
quantification_pfas_default = quantification_pfas_default[
    quantification_pfas_default['Component Name'].isin(pfas_components_msms)
    ]

# initialize new columns
quantification_pfas_default[[
    'RTS',
    'Sample Quantity', 'Sample Unit', 'BDL',
    'IDA Area', 'IDA Concentration', 'IPS Name', 'IPS Area', 'IPS Concentration',
    'Recovery Rate', 'Poor Recovery',
    'TOF Area', 'IAR Deviation', 'Poor IARD',
]] = np.nan

for row_index in quantification_pfas_default.index:
    sample_index = quantification_pfas_default.loc[row_index, 'Sample Index']
    sample_name = quantification_pfas_default.loc[row_index, 'Sample Name']
    ida_channel_name = quantification_pfas_default.loc[row_index, 'IDA Name']
    component_name = quantification_pfas_default.loc[row_index, 'Component Name']

    rt_row = retention_time_data[(
        (retention_time_data['Component Name'] == component_name) &
        (retention_time_data['Sample Index'] == sample_index)
    )]
    
    bdl_row = concentration_data[(
        (concentration_data['Component Name'] == component_name) &
        (concentration_data['Sample Index'] == sample_index)
    )]

    ida_row = quantification_ida[(
            (quantification_ida['Component Name'] == ida_channel_name) &
            (quantification_ida['Sample Index'] == sample_index) &
            (quantification_ida['Sample Name'] == sample_name)
    )]

    recovery_row = recovery_table[(
        (recovery_table['Component Name'] == ida_channel_name) &
        (recovery_table['Sample Index'] == sample_index)
    )]

    iar_row = ion_abundance_ratio[(
            (ion_abundance_ratio['Component Name'] == component_name) &
            (ion_abundance_ratio['Sample Index'] == sample_index)
    )]

    # assign Sample Quantity and Sample Unit
    quantification_pfas_default.loc[row_index, 'Sample Quantity'] = \
        sample_input_data.loc[sample_index, 'volume/weight/number of samples']
    quantification_pfas_default.loc[row_index, 'Sample Unit'] = \
        sample_input_data.loc[sample_index, 'unit (e.g. g/mL/sample)']

    # assign RTS
    quantification_pfas_default.loc[row_index, 'RTS'] = rt_row.loc[:,'RTS'].values[0]

    # assign BDL
    quantification_pfas_default.loc[row_index, 'BDL'] = bdl_row.loc[:,'BDL'].values[0]

    # assign IDA Area and Concentration, IPS Name, Area and Concentration, as well as Recovery Rates
    if ida_channel_name in quantification_ida['Component Name'].to_list():
        # quantification_pfas_default.loc[row_index,'IDA Area'] = ida_row.loc[:,'Area'].values[0]
        # quantification_pfas_default.loc[row_index,'IDA Concentration'] = ida_row.loc[:,'Actual Concentration'].values[0]
        # quantification_pfas_default.loc[row_index,'IPS Area'] = ida_row.loc[:,'IPS Area'].values[0]
        # quantification_pfas_default.loc[row_index,'IPS Concentration'] = ida_row.loc[:,'IPS Concentration'].values[0]
        # quantification_pfas_default.loc[row_index, 'IPS Name'] = \
        #     ida_row.loc[:,'Component Group Name'].values[0]
        quantification_pfas_default.loc[row_index, 'Recovery Rate'] = \
            recovery_row.loc[:, 'Recovery Rate'].values[0]
        quantification_pfas_default.loc[row_index, 'Poor Recovery'] = \
            recovery_row.loc[:, 'Poor Recovery'].values[0]

    # quantification_pfas_default.loc[row_index, 'TOF Area'] = \
    #     iar_row.loc[:, 'TOF Area'].values[0]
    quantification_pfas_default.loc[row_index, 'IAR Deviation'] = \
        iar_row.loc[:, 'Ion Abundance Ratio Deviation [%]'].values[0]
    quantification_pfas_default.loc[row_index, 'Poor IARD'] = \
        iar_row.loc[:, f'Ion Abundance Ratio Deviation > {allowed_ion_abundance_deviation}'].values[0]

quantification_pfas_default['Recovery Rate'] = quantification_pfas_default['Recovery Rate'].round(decimals=1)

# Outputs
The following code block writes final concentration table with all information to excel:
- It uses concentration values and indicates all values below detection limit with 'BDL' \
- It uses retention time difference between compound and related internal standard and indicates all values above the allowed difference with 'RTIS = time difference in minutes'
- It uses retention time difference between compound and compound in the calibration midpoint and indicates all values above the allowed difference with 'RTCM = time difference in minutes'
- It uses ion abundance ratio deviations and indicates all values below the allowed deviation with 'IARD = deviation value' \
- It uses recovery rates and indicates all values below or above the indicated thresholds with 'RR = deviation value'. \
In addition, concentration values are converted to ng\g or ng\l respectively.

Writes final long format table to csv

In [ ]:
def flag_values_string(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """Flags column values (most probably concentrations) with channel ratio, recovery rates and MDLs and replaces them with descriptive strings.

    :param data: data frame containing columns 'RTS', 'BDL', 'Poor Recovery', and 'IAR Deviation',
    as well as the column you indicated.
    :type data: pd.DataFrame
    :param column: Colun name of data frame to be filtered or flagged.
    :type column: str
    :return: Data frame, where the column data is flagged.
    :rtype: pd.DataFrame
    """
    final_table = data[['Sample Name', 'Sample Index', 'Component Name', column]]
    final_table[column] = final_table[column].round(decimals=4)
    for (row_index, row) in data.iterrows():
        if np.isnan(row[column]):
            row_conc = 'ND'
        else:
            row_conc = str(final_table.loc[row_index, column])
            if row['RTS'] == True:
                row_conc = 'RTS'
            elif row['BDL'] == True:
                row_conc = 'BDL'
            elif row['Poor Recovery'] == True:
                row_conc ='PRR'
            elif row['Poor IARD'] == True:
                row_conc = 'IARD'
            else:
                row_conc = str(round(row[column],4))
        final_table.loc[row_index, column] = row_conc
    return final_table

def flag_values_na(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """Flags column values (most probably concentrations) which do not meet QAQC criteria,
    and replaces them with NaNs.

    :param data: data frame containing columns RTS', 'BDL', 'Poor Recovery', and 'IAR Deviation',
    as well as the column you indicated.
    :type data: pd.DataFrame
    :param column: Column name of data frame to be filtered or flagged.
    :type column: str
    :return: Data frame, where the column data is flagged.
    :rtype: pd.DataFrame
    """
    final_table = data[['Sample Name', 'Sample Index', 'Component Name', column]]
    final_table[column] = final_table[column].round(decimals=4)
    for (row_index, row) in data.iterrows():
        if row['RTS'] or row['BDL'] or row['Poor Recovery'] or \
            row['Poor IARD']:
                final_table.loc[row_index, column] = np.nan
    return final_table

# Transform concentration to ng/g or ng/l, depending on your sample_unit
quantification_pfas_default[f'Concentration in ng per Unit'] = quantification_pfas_default['Calculated Concentration'] \
    / quantification_pfas_default['Sample Quantity']

# Flag concentration values with channel ratio, recovery rates and detection threshold.
calculated_concentration_string = flag_values_string(data=quantification_pfas_default, column=f'Concentration in ng per Unit')
calculated_concentration_nan = flag_values_na(data=quantification_pfas_default, column=f'Concentration in ng per Unit')

# Pivot concentration tables.
calculated_concentration_string = calculated_concentration_string.pivot_table(
    index=('Sample Index',), columns='Component Name', values=f'Concentration in ng per Unit', aggfunc='first', dropna=False,
)
calculated_concentration_string = calculated_concentration_string[pfas_components_msms]

calculated_concentration_nan = calculated_concentration_nan.pivot_table(
    index=('Sample Index',), columns='Component Name', values=f'Concentration in ng per Unit', aggfunc='first', dropna=False,
)
calculated_concentration_nan = calculated_concentration_nan[pfas_components_msms]

# add unit to index of final concentration table
tuple_index = []
for sample_index in calculated_concentration_string.index:
    tuple_index.append((index_mapper_publication[sample_index], 'ng/' + unit_mapper[sample_index]))

calculated_concentration_string.index = pd.Index(tuple_index)
calculated_concentration_nan.index = pd.Index(tuple_index)

# Write pivot tables to existing excel file
with pd.ExcelWriter(processed_filepath_xlsx, engine='openpyxl', mode='a') as writer:
    calculated_concentration_string.to_excel(writer, sheet_name=f'Concentration (ng per Unit)')
    calculated_concentration_nan.to_excel(writer, sheet_name=f'Numeric Conc. (ng per Unit)')

# Write long format data to csv
quantification_pfas_default = quantification_pfas_default[[
    'Sample Name', 'Sample Index', 'Acquisition Date & Time','Component Name',
    'Calculated Concentration', 'Concentration in ng per Unit', 'Sample Unit',
    'RTS', 'BDL', 'Poor Recovery', 'Poor IARD',
]]
quantification_pfas_default.rename(columns={'Calculated Concentration': 'Mass in ng'}, inplace=True)
quantification_pfas_default.sort_values(by=['Sample Index', 'Component Name'], inplace=True)
quantification_pfas_default.index = [i for i in range(len(quantification_pfas_default))]

# rename samples to publication names
quantification_pfas_default['Sample Name'] = [index_mapper_publication[sample_index] for sample_index in quantification_pfas_default['Sample Index'].to_list()]

quantification_pfas_default.to_csv(processed_filepath_csv)

# Print output
display(calculated_concentration_string)